# ResilioChain — EDA | Session 2, Part B: Univariate Analysis

---

## What is Univariate Analysis?

Univariate means **one variable at a time**.  
Before comparing columns or building models, we examine each column individually — its shape, spread, skewness, and outliers.  
This session answers the most basic but most important question: **what does each column actually look like?**

---

## Sections in This Notebook

| Section | Focus |
|---------|-------|
| A | `closing_stock` — distribution, shape, bucketing, per-product stats |
| B | `stockout_flag` — class balance, per-product rates, data integrity |
| C | `suppliers.csv` — lead time and reliability distributions |
| D | Connecting both files — supplier vs stockout outcomes |

---

## Setup — Imports and Data Load

In [18]:
import numpy as np
import pandas as pd

inv = pd.read_csv('../../data/inventory_clean.csv')
sup = pd.read_csv('../../data/suppliers.csv')

inv['date'] = pd.to_datetime(inv['date'])

print("inventory_clean shape:", inv.shape)
print("suppliers shape:      ", sup.shape)
print(inv.head())

inventory_clean shape: (1825, 5)
suppliers shape:       (3, 4)
        date product_id  closing_stock  stockout_flag supplier_id
0 2024-01-01       P001            585              0        S001
1 2024-01-02       P001            571              0        S001
2 2024-01-03       P001            557              0        S001
3 2024-01-04       P001            545              0        S001
4 2024-01-05       P001            526              0        S001


---

## Section A — Analysing `closing_stock`

### Q1 — Basic Distribution of `closing_stock`

**Problem:**  
Before any modelling or grouping, we need to understand how `closing_stock` behaves as a single variable across all 1825 rows.  
`.describe()` gives us eight numbers that together tell the full story of a column's spread, centre, and extremes.  
We then compare **mean vs median** — this single comparison tells us whether the distribution leans toward high or low values.

In [19]:
print(inv['closing_stock'].describe())
print()
print("Mean:  ", inv['closing_stock'].mean().round(2))
print("Median:", inv['closing_stock'].median())
print("Difference (mean - median):", 
      round(inv['closing_stock'].mean() - inv['closing_stock'].median(), 2))

count    1825.000000
mean      212.640000
std       143.644802
min         0.000000
25%        93.000000
50%       211.000000
75%       327.000000
max       595.000000
Name: closing_stock, dtype: float64

Mean:   212.64
Median: 211.0
Difference (mean - median): 1.64


**Results:**

| Statistic | Value  |
|-----------|--------|
| Mean      | 212.64 |
| Median    | 211.00 |
| Std       | 143.64 |
| Min       | 0      |
| Max       | 595    |

**Mean vs Median:** Mean (212.64) is slightly higher than median (211.00) — a difference of only 1.64.

The two values are almost identical, which means the distribution is very close to symmetric.  
There is a tiny rightward pull from large restocking deliveries on some days — but not enough to create meaningful skew.  
The warehouse does not systematically over-stock or under-stock at an overall level.  
**The real problem is hidden at the product level — not visible in the overall average.**

---

### Q2 — Shape of the Distribution: Skewness and Kurtosis

**Problem:**  
`.describe()` tells us where the data sits but not the **shape** of it.  
Two distributions can share the same mean and std but look completely different.  
- **Skewness** measures which direction the tail is pulled.  
- **Kurtosis** measures whether the peak is sharp or flat.  
Both numbers together describe the full shape of `closing_stock`.

In [20]:
skewness = inv['closing_stock'].skew()
kurtosis = inv['closing_stock'].kurtosis()

print("Skewness:", round(skewness, 4))
print("Kurtosis:", round(kurtosis, 4))

Skewness: 0.174
Kurtosis: -0.8386


**Results:**

| Measure  | Value   | Category |
|----------|---------|----------|
| Skewness | 0.1740  | Approximately symmetric — between -0.5 and 0.5 |
| Kurtosis | -0.8386 | Platykurtic — flatter peak than a normal curve |

**Skewness (0.17):** Close to zero — only a slight rightward lean. The distribution is broadly symmetric.

**Kurtosis (-0.84):** Negative value means the distribution is flatter than a normal curve.  
Stock levels are spread broadly across the range rather than clustering tightly around one value.

**In plain English:** `closing_stock` is spread fairly evenly across its range with no extreme clustering and only a slight tendency toward higher values on restocking days.

---

### Q3 — Bucketing `closing_stock` into Ranges Using `pd.cut()`

**Problem:**  
A continuous column like `closing_stock` is hard to read row by row.  
Bucketing groups values into fixed ranges so we can see which stock level zone the warehouse spends most of its time in.  
`pd.cut()` is the pandas tool for creating fixed-width buckets.

In [21]:
bins = [0, 100, 200, 300, 400, 500, 600]

inv['stock_bucket'] = pd.cut(inv['closing_stock'], bins=bins)

bucket_counts = inv['stock_bucket'].value_counts().sort_index()
print(bucket_counts)

stock_bucket
(0, 100]      272
(100, 200]    377
(200, 300]    412
(300, 400]    375
(400, 500]    138
(500, 600]     40
Name: count, dtype: int64


**Results:**

| Bucket     | Row Count | Note |
|------------|-----------|------|
| (0, 100]   | 272       | Critically low stock — stockout risk zone |
| (100, 200] | 377       | Low-moderate stock |
| (200, 300] | 412       | Most common range |
| (300, 400] | 375       | Healthy buffer |
| (400, 500] | 138       | High stock — rarely reached |
| (500, 600] | 40        | Near-maximum — very rare |

**Most rows:** (200, 300] with 412 rows — the warehouse most commonly holds 200–300 units.

**Fewest rows:** (500, 600] with only 40 rows — very high stock days are rare.  
The warehouse almost never holds near-maximum inventory.

**Operational concern:** The (0, 100] bucket has 272 rows — 272 product-days where stock was critically low.  
These are the days closest to a stockout. One unexpected demand spike on any of these days would trigger the flag.

---

### Q4 — Per-Product `closing_stock` Statistics

**Problem:**  
The overall `.describe()` from Q1 mixes all five products together.  
A product with extremely low stock can be hidden behind the average of products with healthy stock.  
We need each product's own statistics to identify who is running dangerously lean.

In [22]:
product_stats = inv.groupby('product_id')['closing_stock'].describe().round(2)
print(product_stats)
print()
print("Products with minimum stock = 0:")
print(product_stats[product_stats['min'] == 0].index.tolist())

            count    mean     std   min    25%    50%    75%    max
product_id                                                         
P001        365.0  138.24  143.12   0.0    0.0  101.0  257.0  585.0
P002        365.0  220.71  130.89   0.0  109.0  219.0  325.0  584.0
P003        365.0  250.33  121.96  17.0  150.0  246.0  350.0  583.0
P004        365.0  173.49  145.71   0.0   25.0  162.0  290.0  588.0
P005        365.0  280.43  128.42  30.0  174.0  283.0  383.0  595.0

Products with minimum stock = 0:
['P001', 'P002', 'P004']


**Results:**

| Product | Mean   | Min | Max | Std    |
|---------|--------|-----|-----|--------|
| P001    | 138.24 | 0   | 585 | 143.12 |
| P002    | 220.71 | 0   | 584 | 130.89 |
| P003    | 250.33 | 17  | 583 | 121.96 |
| P004    | 173.49 | 0   | 588 | 145.71 |
| P005    | 280.43 | 30  | 595 | 128.42 |

**Products with minimum = 0:** P001, P002, P004 — these three physically ran out of stock on at least one day.

| Rank | Product | Mean Stock | Status |
|------|---------|------------|--------|
| Highest | P005 | 280.43 | Well buffered all year |
| Lowest  | P001 | 138.24 | Runs dangerously lean |

**P001 has both the lowest mean AND a minimum of 0.**  
It spends most of its days with thin stock and repeatedly hits empty.  
This is the product most in need of a smarter reorder system.

---

### Q5 — Per-Product Skewness of `closing_stock`

**Problem:**  
The overall skewness from Q2 averages across all five products.  
Individual products may have very different distribution shapes.  
A product with strong right skew spends most days at low stock but occasionally spikes high — a warning sign for stockout risk.

In [23]:
product_skew = inv.groupby('product_id')['closing_stock'].skew().round(4)
print(product_skew.sort_values(ascending=False))

product_id
P001    0.7090
P004    0.4562
P002    0.1955
P003    0.1068
P005    0.0971
Name: closing_stock, dtype: float64


**Results (ranked most to least right-skewed):**

| Product | Skewness | Shape |
|---------|----------|-------|
| P001    | 0.7090   | Right skewed |
| P004    | 0.4562   | Mild right skew |
| P002    | 0.1955   | Approximately symmetric |
| P003    | 0.1068   | Approximately symmetric |
| P005    | 0.0971   | Approximately symmetric |

**Most right-skewed: P001 (0.709)**

Right skew in closing stock means: most days P001 sits at **low stock levels**, but occasionally a large restocking delivery pushes it high.  
The distribution has a long right tail from those spike days — but the bulk of the data clusters on the low side.

**Connection to the grouping session:**  
P001 had 125 stockout days and the lowest mean closing stock.  
The skew confirms the pattern — P001 spends most of its time running low, with infrequent restocking events that temporarily push it up.  
This is the signature of a product with **too-long a reorder gap**.

---

### Q6 — Outlier Detection Using IQR (Interquartile Range)

**Problem:**  
Outliers are data points that sit far outside the normal range.  
In stock data, outliers could be genuine operational events (a massive delivery, an emergency stockout) or data errors.  
The **IQR method** defines what "too far outside" means mathematically, without assuming any particular distribution shape.

In [24]:
Q1 = inv['closing_stock'].quantile(0.25)
Q3 = inv['closing_stock'].quantile(0.75)
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print("Q1 (25th percentile):", Q1)
print("Q3 (75th percentile):", Q3)
print("IQR:                 ", IQR)
print("Lower fence:         ", lower_fence)
print("Upper fence:         ", upper_fence)
print()

outliers = inv[(inv['closing_stock'] < lower_fence) | 
               (inv['closing_stock'] > upper_fence)]

print("Outlier count:", len(outliers))

Q1 (25th percentile): 93.0
Q3 (75th percentile): 327.0
IQR:                  234.0
Lower fence:          -258.0
Upper fence:          678.0

Outlier count: 0


**Results:**

| Measure        | Value  |
|----------------|--------|
| Q1 (25th pct)  | 93     |
| Q3 (75th pct)  | 327    |
| IQR            | 234    |
| Lower Fence    | -258   |
| Upper Fence    | 678    |
| Outliers Found | 0      |

**Zero outliers detected.**

The IQR fences are very wide (lower = -258, upper = 678) because the data itself is widely spread (std = 143.64).  
Since `closing_stock` cannot go below 0 and the max is only 595 — well within the upper fence of 678 — no values fall outside.

**What this tells us:**  
`closing_stock` has no extreme anomalous values.  
The variation we see is **genuine operational variation**, not data errors or recording mistakes.  
Every value in the column is within a reasonable statistical range of the rest of the data.

---

## Section B — Analysing `stockout_flag`

### Q7 — Class Distribution of `stockout_flag`

**Problem:**  
`stockout_flag` is binary: 0 or 1. Before using it as a **target variable** in a machine learning model, we must know how balanced the two classes are.  
A heavily imbalanced target causes a model to learn the wrong thing — it predicts the majority class almost always and still shows high accuracy, while being completely blind to the minority class (the stockouts).

In [25]:
flag_counts = inv['stockout_flag'].value_counts()
print(flag_counts)
print()

stockout_rate = (flag_counts[1] / len(inv)) * 100
print("Stockout rate: {:.2f}%".format(stockout_rate))

stockout_flag
0    1614
1     211
Name: count, dtype: int64

Stockout rate: 11.56%


**Results:**

| Class | Label       | Count | Percentage |
|-------|-------------|-------|------------|
| 0     | No stockout | 1614  | 88.44%     |
| 1     | Stockout    | 211   | 11.56%     |

**Overall stockout rate: 11.56%**  
The no-stockout class appears **7.6x more** than the stockout class — this is a class imbalance.

**Why this matters for modelling:**  
A classifier trained on this data without correction will learn to predict 0 (no stockout) almost every time and still achieve 88% accuracy — because that is the majority class.  
It will appear to work well but will almost never catch an actual stockout, which is the entire point of the model.

Class imbalance must be handled using techniques like **SMOTE**, **class weighting**, or **threshold tuning** before any model is trained on this target variable.

---

### Q8 — Stockout Rate per Product

**Problem:**  
The overall 11.56% rate from Q7 masks which products are responsible.  
A single catastrophically unreliable product can inflate the overall rate while others are perfectly fine.  
We need per-product rates to separate the problem products from the healthy ones.

In [26]:
product_stockout = inv.groupby('product_id')['stockout_flag'].agg(
    total_stockout_days='sum',
    stockout_rate='mean'
)

product_stockout['stockout_rate_%'] = (product_stockout['stockout_rate'] * 100).round(2)
product_stockout = product_stockout.drop(columns='stockout_rate')

print(product_stockout.sort_values('stockout_rate_%', ascending=False))

            total_stockout_days  stockout_rate_%
product_id                                      
P001                        125            34.25
P004                         76            20.82
P002                         10             2.74
P003                          0             0.00
P005                          0             0.00


**Results:**

| Product | Total Stockout Days | Stockout Rate |
|---------|---------------------|---------------|
| P001    | 125                 | 34.25%        |
| P004    | 76                  | 20.82%        |
| P002    | 10                  | 2.74%         |
| P003    | 0                   | 0.00%         |
| P005    | 0                   | 0.00%         |

**Products with exactly 0% stockout rate:** P003 and P005  
**Product above 30%:** P001 at 34.25%

P001 is out of stock on **more than 1 in every 3 days** across the entire year.  
For a warehouse product, this is a critical reliability failure.  
Any downstream process depending on P001 faces disruption on a third of all working days.  
**This product cannot be considered reliably available under the current supply chain setup.**

---

### Q9 — Zero Stock Days per Product and Cross-Check with Q8

**Problem:**  
`stockout_flag` was set by whoever generated the data.  
`closing_stock = 0` is an independently measurable fact from the raw numbers.  
If these two agree perfectly — every day with stock = 0 also has flag = 1 — the dataset is internally consistent.  
If they disagree, we have a data quality problem that would corrupt any model trained on it.  
**This cross-check is mandatory before any modelling begins.**

In [27]:
zero_stock_days = (inv[inv['closing_stock'] == 0]
                   .groupby('product_id')
                   .size()
                   .rename('zero_stock_days'))

print("Zero stock days per product:")
print(zero_stock_days)

print()
print("Stockout days from Q8 (for comparison):")
print(inv.groupby('product_id')['stockout_flag'].sum())

Zero stock days per product:
product_id
P001    125
P002     10
P004     76
Name: zero_stock_days, dtype: int64

Stockout days from Q8 (for comparison):
product_id
P001    125
P002     10
P003      0
P004     76
P005      0
Name: stockout_flag, dtype: int64


**Results:**

| Product | Zero Stock Days | Stockout Days (flag=1) | Match |
|---------|-----------------|------------------------|-------|
| P001    | 125             | 125                    | Yes   |
| P002    | 10              | 10                     | Yes   |
| P004    | 76              | 76                     | Yes   |
| P003    | 0               | 0                      | Yes   |
| P005    | 0               | 0                      | Yes   |

**Perfect match across all five products.**

Every day where `closing_stock = 0` has a corresponding `stockout_flag = 1`.  
No missed stockouts, no false alarms. The two columns are in complete agreement.

This confirms the dataset is **internally consistent** and that `stockout_flag` was derived directly and correctly from `closing_stock`.  
We can trust either column as a reliable signal in downstream analysis and modelling.

---

## Section C — Analysing `suppliers.csv`

### Q10 — Distribution of `lead_time_days` Across Suppliers

**Problem:**  
`suppliers.csv` only has 3 rows. Traditional `.describe()` statistics are less meaningful at this scale.  
But the **coefficient of variation (CV)** — std divided by mean — is still useful:  
it tells us how different the three suppliers are from each other **relative to their average**.  
A high CV means supplier choice is a high-stakes decision.

In [28]:
print(sup['lead_time_days'].describe())
print()

cv = (sup['lead_time_days'].std() / sup['lead_time_days'].mean()) * 100
print("Coefficient of Variation: {:.2f}%".format(cv))

count     3.000000
mean      8.000000
std       5.567764
min       3.000000
25%       5.000000
50%       7.000000
75%      10.500000
max      14.000000
Name: lead_time_days, dtype: float64

Coefficient of Variation: 69.60%


**Results:**

| Statistic | Value  |
|-----------|--------|
| Mean      | 8.00 days |
| Min       | 3 days |
| Max       | 14 days |
| Std       | 5.57 days |
| CV        | 69.60% |

**CV of 69.60% is extremely high** — the three suppliers are very different from each other in delivery speed.

The gap between the fastest (3 days) and slowest (14 days) is **4.7x**.  
Assigning a product to AsiaTech vs LocalFast is not a marginal decision — it is the difference between a **3-day recovery** from a stockout and a **14-day wait**.

Supplier assignment is one of the most important decisions in this supply chain and must be treated as a **high-impact variable** in any reorder model.

---

### Q11 — Distribution of `reliability` Across Suppliers

**Problem:**  
We apply the same analysis to reliability.  
The key question: compared to `lead_time_days`, does reliability vary **more or less** across the three suppliers?  
The answer tells us which dimension — speed or consistency — creates more differentiation between suppliers.

In [29]:
print(sup['reliability'].describe())
print()

gap = sup['reliability'].max() - sup['reliability'].min()
gap_pct = (gap / sup['reliability'].max()) * 100

print("Gap (max - min):          ", round(gap, 4))
print("Gap as % of highest score: {:.2f}%".format(gap_pct))
print()
print("Lead time std:   ", round(sup['lead_time_days'].std(), 4))
print("Reliability std: ", round(sup['reliability'].std(), 4))

count    3.000000
mean     0.960000
std      0.036056
min      0.920000
25%      0.945000
50%      0.970000
75%      0.980000
max      0.990000
Name: reliability, dtype: float64

Gap (max - min):           0.07
Gap as % of highest score: 7.07%

Lead time std:    5.5678
Reliability std:  0.0361


**Results:**

| Statistic | Value  |
|-----------|--------|
| Mean      | 0.960  |
| Min       | 0.920  |
| Max       | 0.990  |
| Std       | 0.0361 |
| Gap (max - min) | 0.07 (7.07% of highest score) |

**Compared to `lead_time_days` (std = 5.57), reliability has a much smaller std (0.036).**  
Lead time varies far more across suppliers than reliability does.

The 0.07 gap between best (0.99) and worst (0.92) reliability seems small — but operationally:

- AsiaTech fails **8% of the time** vs LocalFast's **1% of the time**
- That is **8x more late deliveries**
- Combined with a 14-day lead time, each failure leaves the warehouse waiting two weeks for a replacement

**The suppliers appear broadly similar in reliability scores but are meaningfully different in their operational impact when they fail.**

---

## Section D — Connecting Both Files

### Q12 — Lead Time, Reliability, and Actual Stockouts Side by Side

**Problem:**  
`suppliers.csv` tells us what each supplier's reliability score **claims**.  
`inventory_clean.csv` tells us what **actually happened**.  
Putting both side by side is the only way to answer:  
does the reliability score actually predict real-world stockout outcomes?  
If it does, we can use it as a model feature. If it does not, it is misleading.

In [30]:
merged = inv.merge(sup, on='supplier_id', how='left')

stockout_by_supplier = merged.groupby('name')['stockout_flag'].sum().reset_index()
stockout_by_supplier.columns = ['name', 'total_stockout_days']

summary = stockout_by_supplier.merge(
    sup[['name', 'lead_time_days', 'reliability']],
    on='name'
).sort_values('lead_time_days')

print(summary.to_string(index=False))

               name  total_stockout_days  lead_time_days  reliability
LocalFast Supply Co                    0               3         0.99
      EuroGoods Ltd                   10               7         0.97
   AsiaTech Imports                  201              14         0.92


**Results:**

| Supplier            | Lead Time | Reliability | Total Stockout Days |
|---------------------|-----------|-------------|---------------------|
| LocalFast Supply Co | 3 days    | 0.99        | 0                   |
| EuroGoods Ltd       | 7 days    | 0.97        | 10                  |
| AsiaTech Imports    | 14 days   | 0.92        | 201                 |

**Faster suppliers are more reliable — the pattern is perfectly monotonic.**  
As lead time increases, reliability decreases and stockout days increase. No exceptions.

The reliability score **does** predict real stockout outcomes — the ranking matches exactly.  
However, the score **understates the severity**:  
AsiaTech's 0.92 sounds reasonable but its products account for **201 of the 211 total stockout days — 95% of all stockout damage**.

A single percentage point of reliability difference translates into a massive real-world outcome difference when combined with a long lead time.

---

### Q13 — Does Supplier Lead Time Explain Stockout Frequency?

**Problem:**  
We isolate the relationship between `lead_time_days` and stockouts at the supplier level in a clean side-by-side table.  
This is the **direct test of ResilioChain's core hypothesis:** longer lead time = more stockouts.

In [31]:
q13 = merged.groupby('supplier_id')['stockout_flag'].sum().reset_index()
q13.columns = ['supplier_id', 'total_stockout_days']

q13 = q13.merge(sup[['supplier_id', 'lead_time_days', 'reliability']], 
                on='supplier_id')

print(q13.sort_values('lead_time_days').to_string(index=False))

supplier_id  total_stockout_days  lead_time_days  reliability
       S003                    0               3         0.99
       S002                   10               7         0.97
       S001                  201              14         0.92


**Results:**

| Supplier | Lead Time | Reliability | Total Stockout Days |
|----------|-----------|-------------|---------------------|
| S003     | 3 days    | 0.99        | 0                   |
| S002     | 7 days    | 0.97        | 10                  |
| S001     | 14 days   | 0.92        | 201                 |

**Yes — longer lead time always means more stockout days.** The relationship is perfectly monotonic with no exceptions.

S001's 14-day lead time creates a situation where any delay or demand spike cannot be recovered quickly.  
By the time a reorder arrives, the shelf has already been empty for days.  
**ResilioChain's core value is shortening this reaction window.**

---

### Q14 — Stock Level Profile per Supplier

**Problem:**  
Stockout days tell us **when** a product hit zero.  
Mean closing stock tells us the **average daily health** of inventory.  
A supplier whose products consistently sit at low average stock is a structural risk — always one bad week away from hitting zero.  
We need mean, min, and max together to see the full stock level profile per supplier.

In [32]:
q14 = (merged.groupby('name')['closing_stock']
       .agg(mean_stock='mean', min_stock='min', max_stock='max')
       .round(2)
       .reset_index())

q14 = q14.merge(sup[['name', 'lead_time_days']], on='name')
print(q14.sort_values('mean_stock').to_string(index=False))

               name  mean_stock  min_stock  max_stock  lead_time_days
   AsiaTech Imports      155.87          0        588              14
LocalFast Supply Co      250.33         17        583               3
      EuroGoods Ltd      250.57          0        595               7


**Results:**

| Supplier            | Mean Stock | Min Stock | Max Stock | Lead Time |
|---------------------|------------|-----------|-----------|-----------|
| AsiaTech Imports    | 155.87     | 0         | 588       | 14 days   |
| LocalFast Supply Co | 250.33     | 17        | 583       | 3 days    |
| EuroGoods Ltd       | 250.57     | 0         | 595       | 7 days    |

**Supplier with lowest mean closing stock: AsiaTech Imports (155.87)**

AsiaTech's products (P001 and P004) run at an average of 155.87 units — significantly lower than the other two suppliers who average around 250 units.  
Their minimum hitting 0 confirms they physically run dry.

**Yes, the lead time explains it directly.**  
With 14 days between order and delivery, the warehouse must place orders much earlier to avoid running out.  
If the reorder point is set too low — or a delivery is late even once — the product empties out and stays empty for up to two weeks.  
**AsiaTech's long lead time structurally forces their products to run at higher risk.**

---

### Q15 — Full Univariate Summary

*This cell synthesises findings from Q1 to Q14. No new code.*

---

**1. Overall shape and spread of `closing_stock`**

`closing_stock` across 1825 rows is approximately symmetric (skewness = 0.17), flat relative to a normal distribution (kurtosis = -0.84), and widely spread (std = 143.64).  
The warehouse operates across a broad stock range with no extreme outliers.  
The variation is genuine and operational — not noise or data error.

---

**2. Products most at risk**

| Product | Stockout Rate | Status |
|---------|---------------|--------|
| P001    | 34.25%        | Critical — out of stock 1 in 3 days |
| P004    | 20.82%        | High risk |
| P002    | 2.74%         | Minor |
| P003    | 0.00%         | Safe |
| P005    | 0.00%         | Safe |

P001 and P004 are both supplied by AsiaTech Imports. The supplier is the root cause.

---

**3. Do supplier metrics reflect real inventory outcomes?**

Yes — perfectly. The ranking by reliability score (0.92, 0.97, 0.99) matches the ranking by actual stockout days (201, 10, 0) exactly.  
However, the scores **understate the severity**. AsiaTech's 0.92 score sounds acceptable but it generates **95% of all stockout damage**.  
Reliability score alone is not a sufficient metric — it must be combined with lead time to assess true supplier risk.

---

**4. Single most important finding for the forecasting model**

**Lead time is the dominant structural risk factor.**  
Products on a 14-day lead time have a 34% stockout rate. Products on a 3-day lead time have 0%.  
The model must treat `lead_time_days` as a high-priority feature and must account for the **88%/12% class imbalance** in the target variable — or it will fail to predict the very stockouts it was built to prevent.